# Climate Resilience & Urban "Heat Island" (UHI) Vulnerability Analytics
**Course**: B.Tech Computer Science & Engineering (3rd Year) — Data Visualization Laboratory
**Student Author**: Vedant
**Data Sources**: NASA Landsat 8/9 TIRS (Thermal Infrared), Copernicus Sentinel-2 MSI (NDVI), OpenStreetMap, Municipal GIS & Census Demographics

---

## 1. Project Background & Research Problem
Urban concrete density, asphalt paving, and low albedo cause metropolitan neighborhoods to absorb and re-radiate solar thermal energy, creating **Urban Heat Islands (UHI)** where surface temperatures exceed surrounding rural baselines by up to $+8.4^\circ\text{C}$.

### Visual Story & Analytical Objectives:
1. **Spatial Micro-Climate Distribution**: Overlay satellite Land Surface Temperature (LST) with satellite Normalized Difference Vegetation Index (NDVI) via interactive Leaflet/Folium mapping.
2. **Canopy vs. Temperature Regression**: Quantify cooling potential through Ordinary Least Squares (OLS) regression between Tree Canopy (%) and Surface Temperature ($^\circ\text{C}$).
3. **Critical Energy Infrastructure**: Assess longitudinal municipal grid strain by correlating Extreme Heat Days ($T \ge 35^\circ\text{C}$) against Summer Peak Grid Load (MW).
4. **Environmental Justice**: Demonstrate how historical urban planning and socioeconomic disparity expose low-income communities to disproportionate heat illness risks.

### Step 1: Import Scientific & Visualization Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import folium
from folium.plugins import HeatMap

# Set publication styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8
print("Libraries successfully loaded!")

### Step 2: Ingest Multi-Source Satellite & Demographic Datasets

In [ ]:
zones_df = pd.read_csv('data/uhi_neighborhood_metrics.csv')
time_df = pd.read_csv('data/uhi_timeseries_energy_grid.csv')

print(f"Loaded {len(zones_df)} urban zones across 16 biophysical and socioeconomic dimensions.")
print(f"Loaded {len(time_df)} years of longitudinal energy grid stress observations.")
display(zones_df.head(4))

### Step 3: Data Quality Audit & Feature Engineering
We compute the **Heat Vulnerability Index (HVI)** using standardized z-scores across Exposure, Sensitivity, and Adaptive Capacity.

In [ ]:
print("=== Missing Values Check ===")
print(zones_df.isnull().sum())

# Summary statistics for critical biophysical attributes
display(zones_df[['surface_temp_c', 'uhi_intensity_c', 'tree_canopy_pct', 'ndvi_index', 'median_income_usd', 'heat_hospitalizations_per_10k']].describe().round(2))

### Step 4: Key Visualization 1 — Spatial Micro-Climate & NDVI Map (Folium)

In [ ]:
center_lat = zones_df['latitude'].mean()
center_lon = zones_df['longitude'].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles='OpenStreetMap')

fg_temp = folium.FeatureGroup(name='🔥 Surface Temperature (°C)', show=True)
for _, r in zones_df.iterrows():
    color = '#2ca02c' if r['surface_temp_c'] < 30 else '#ff7f0e' if r['surface_temp_c'] < 35 else '#d62728' if r['surface_temp_c'] < 40 else '#7f0000'
    popup_html = f"<b>{r['zone_name']}</b><br>LST: {r['surface_temp_c']}°C<br>Canopy: {r['tree_canopy_pct']}%<br>Income: ${r['median_income_usd']:,}"
    folium.CircleMarker(
        location=[r['latitude'], r['longitude']],
        radius=8 + (r['surface_temp_c'] - 24) * 0.7,
        color=color, fill=True, fill_color=color, fill_opacity=0.8,
        popup=folium.Popup(popup_html, max_width=250)
    ).add_to(fg_temp)

fg_temp.add_to(m)
folium.LayerControl().add_to(m)
m.save('visualizations/fig1_spatial_folium_map.html')
print("Interactive Folium map saved to visualizations/fig1_spatial_folium_map.html")

### Step 5: Key Visualization 2 — OLS Regression (Tree Canopy vs. Surface Temp)

In [ ]:
plt.figure(figsize=(9, 5.5))
x = zones_df['tree_canopy_pct']
y = zones_df['surface_temp_c']

slope, intercept, r_val, p_val, std_err = stats.linregress(x, y)

sns.scatterplot(data=zones_df, x='tree_canopy_pct', y='surface_temp_c', hue='vulnerability_tier', s=80, alpha=0.9, palette='tab10')
x_fit = np.linspace(x.min(), x.max(), 100)
plt.plot(x_fit, slope * x_fit + intercept, color='#222222', linestyle='--', linewidth=2, label=f'OLS: T = {intercept:.1f} - {abs(slope):.2f}*Canopy%')

plt.title(f'Tree Canopy Mitigation: R² = {r_val**2:.2f}, Cooling Rate = {abs(slope):.2f}°C per 1% Canopy', fontsize=12, weight='bold')
plt.xlabel('Tree Canopy Coverage (%)')
plt.ylabel('Land Surface Temperature (°C)')
plt.legend()
plt.tight_layout()
plt.savefig('visualizations/fig2_canopy_vs_temperature_regression.png', dpi=300)
plt.show()

### Step 6: Key Visualization 3 — Dual-Axis Grid Load vs. Extreme Heat Days

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
years = time_df['year']
bars = ax1.bar(years, time_df['summer_peak_grid_load_mw'], color='#3182bd', width=0.55, alpha=0.85, label='Peak Grid Demand (MW)')
ax1.set_ylabel('Peak Grid Load (MW)', color='#1c5682', weight='bold')
ax1.set_ylim(6000, 10500)

ax2 = ax1.twinx()
line = ax2.plot(years, time_df['extreme_heat_days_count'], color='#de2d26', marker='o', linewidth=2.8, label='Days ≥ 35°C')
ax2.set_ylabel('Extreme Heat Days', color='#de2d26', weight='bold')
ax2.grid(False)

plt.title('Urban Grid Strain: Peak Summer Load (MW) vs. Annual Extreme Heat Days', weight='bold')
fig.tight_layout()
plt.savefig('visualizations/fig3_dual_axis_energy_vs_heat_days.png', dpi=300)
plt.show()